# 🎬 MiniMax-H3 Diário

**Dois cliques no Colab. Depois, só o painel.**

1. Use seu runtime **80 GB GPU + 167 GB RAM** e execute **Instalar**.
2. Execute **Abrir painel**, autorize o Drive e clique no link.
3. No painel, clique em **Preparar**. Envie imagens, escreva o prompt e clique **Gerar vídeo**.

**Primeira vez:** precisa baixar e carregar os pesos (~141 GiB + LoRAs). Isso leva vários minutos.
**Uso diário:** com o Drive ativado, os pesos ficam salvos; em sessões seguintes são copiados para
o disco rápido e carregados uma vez. Durante a sessão os dois modelos permanecem na RAM.
O condicionador e o gerador usam a GPU em momentos separados; as VAEs usam offload com reserva de memória.
Prompts/imagens já preparados são reutilizados para gerar variações com outras seeds.

O Drive precisa de aproximadamente **150 GB livres**. Se não tiver esse espaço, desmarque `USE_DRIVE`:
os pesos continuam disponíveis durante a sessão, mas serão perdidos ao encerrá-la.

**Painel simples:** imagem inicial + final opcionais, prompt, formato, duração, Gerar, prévia e downloads.
Passos/seed/Turbo/variações ficam em opções avançadas. Padrão: **larry · INT8 · 8 passos · Context-IR**.
Vídeos são salvos automaticamente. Não precisa baixar o mesmo modelo a cada vídeo.

O notebook atual permanece separado. Use esta versão em uma sessão livre, sem os modelos do outro notebook na GPU.
Não há garantia de OOM zero em todas as resoluções/durações; o formato rápido reduz a carga.
Para economizar créditos, desligue o runtime quando terminar seu trabalho do dia.


In [1]:
#@title 1 · Instalar (uma vez por sessão)
import subprocess, sys
requirements = ["diffusers==0.40.0", "transformers==5.8.0", "accelerate>=1.14,<2",
                "huggingface-hub==1.24.0", "safetensors>=0.8.0", "gradio==6.29.1",
                "av", "einops", "protobuf", "psutil", "pillow"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", *requirements], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-cache-dir", "--no-deps",
                "--upgrade", "torchao==0.18.0"], check=True)
# Verifica em processo novo: não usa módulos antigos já importados pelo kernel.
subprocess.run([sys.executable, "-c", "import torch, diffusers, transformers, gradio, torchao; "
    "from torchao.quantization import Int8Tensor, PerRow, Int8WeightOnlyConfig; "
    "from diffusers import TorchAoConfig; "
    "assert diffusers.__version__ == '0.40.0'; "
    "assert transformers.__version__.startswith('5.8'); "
    "assert torchao.__version__ == '0.18.0'; "
    "print('Dependências verificadas. Execute a célula 2.')"], check=True)


CompletedProcess(args=['/usr/bin/python3', '-c', "import torch, diffusers, transformers, gradio, torchao; from torchao.quantization import Int8Tensor, PerRow, Int8WeightOnlyConfig; from diffusers import TorchAoConfig; assert diffusers.__version__ == '0.40.0'; assert transformers.__version__.startswith('5.8'); assert torchao.__version__ == '0.18.0'; print('Dependências verificadas. Execute a célula 2.')"], returncode=0)

In [2]:
#@title 2 · Abrir painel
from pathlib import Path
appdir = Path('/content/h3_diario_app')
appdir.mkdir(parents=True, exist_ok=True)
(appdir / 'prompt_rewrite.py').write_text('"""Context-IR style prompt refinement, on the conditioner\'s own Qwen3-VL.\n\nMiniMax-H3 is trained on a *rewritten* prompt: a structured Context Intermediate Representation, not free text.\nMiniMax\'s own rewriter (`H3-Context-IR`) is a hosted service that is not part of the open release, but the two\nofficial prompting guides specify the target format exactly, and the conditioner this Space already holds is the\nfull 64-layer `Qwen3VLForConditionalGeneration` — language-model head, chat template and all. So the rewrite runs\nhere, on the same weights that are about to encode its output, and costs one extra prefill plus a short decode.\n\nThe split of responsibilities is deliberate:\n\n  * **The instruction line is code.** Every keyframe task opens with a fixed alignment sentence whose only variables\n    are the task, the effective duration to two decimals and the index of the final shot. A language model asked for\n    a template with numbers in it will paraphrase the template and round the numbers, so [`base_instruction_line`]\n    formats it from the request instead. `t2va` has no instruction line at all.\n  * **The descriptive core is the model\'s.** `integrated_multimodal_description` (or, in full-reference mode, the six\n    sections from `subject_definitions` to `non_diegetic_music`) is what actually needs a vision-language model:\n    it has to look at the keyframes or the references and write the timeline they imply.\n\nNothing here is on the encoding path. The refined prompt is fed to the unchanged encoder as `prompt`, and any failure\nfalls back to the raw prompt.\n"""\n\nfrom __future__ import annotations\n\nimport os\nimport re\nimport time\n\nimport numpy as np\nimport torch\nfrom PIL import Image\n\n\n# The conditioner reads a keyframe at the full canvas and a reference image at a 2048 pixel short edge. The *rewriter*\n# does not need that: it needs to see what is in the frame. A 1024 pixel long edge is ~700 vision tokens per image\n# through the 16 pixel patches and the 2x2 merge, which keeps a nine-image request inside a sane prefill.\nREWRITE_IMAGE_LONG_EDGE = 1024\n# A reference video reaches the conditioner at 2 fps, so that is what the rewriter is shown as well — capped, so that\n# one long clip cannot own the prefill. 32 frames is 16 seconds, past the 15 MiniMax-H3 generates.\nREWRITE_VIDEO_FPS = 2.0\nREWRITE_VIDEO_MAX_FRAMES = 32\n\nMINIMAX_H3_FPS = 24\n# The video VAE\'s own chunking, which the frame count has to land on: `17 * n + 5`. Arguments to the library\'s\n# `align_num_frames` since the blocks were refactored, where they used to be module constants of a `packing` module\n# that no longer exists.\nFRAMES_PER_CHUNK, LATENTS_PER_CHUNK = 17, 5\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# The deterministic half: task resolution, duration and the instruction line.\n# ---------------------------------------------------------------------------------------------------------------\n\n\ndef base_task(image, last_image) -> str:\n    """Which of the four base tasks a `/encode` request is, from the keyframes it carries."""\n    if image is not None and last_image is not None:\n        return "fl2va"\n    if image is not None:\n        return "i2va"\n    if last_image is not None:\n        return "l2va"\n    return "t2va"\n\n\ndef aligned_num_frames(num_frames: int) -> int:\n    """The `17 * n + 5` frame count the video VAE decodes, which is what the request actually generates."""\n    from diffusers.modular_pipelines.minimax_h3.modular_pipeline import align_num_frames\n\n    return int(align_num_frames(int(num_frames), FRAMES_PER_CHUNK, LATENTS_PER_CHUNK))\n\n\ndef duration_seconds(num_frames: int) -> float:\n    """The effective duration of the request, at MiniMax-H3\'s fixed 24 fps."""\n    return aligned_num_frames(num_frames) / MINIMAX_H3_FPS\n\n\ndef final_shot_index(core: str) -> int:\n    """The index of the last `[Shot N]` the rewrite actually wrote, which the instruction line points at."""\n    indices = [int(match) for match in re.findall(r"\\[Shot (\\d+)\\]", core)]\n    return max(indices) if indices else 1\n\n\ndef base_instruction_line(task: str, duration: float, final_shot: int) -> str | None:\n    """The alignment instruction a keyframe task opens with, verbatim from the guide with the numbers filled in.\n\n    The bracket styles differ between the three templates — `fl2va` writes `Picture 2 (from Shot 1)` while `l2va`\n    writes `<Picture 1> (from [Shot 1])`. That is how the guide specifies them, so that is what is reproduced.\n\n    Args:\n        task (`str`): `"t2va"`, `"i2va"`, `"fl2va"` or `"l2va"`.\n        duration (`float`): The effective duration in seconds, rendered to exactly two decimals.\n        final_shot (`int`): The index of the final shot of the rewrite, `N` in the templates.\n\n    Returns:\n        `str` or `None`: the instruction line, or `None` for `t2va`, which has none.\n    """\n    if task == "t2va":\n        return None\n    if task == "i2va":\n        return (\n            "For the target video, at 0.00 seconds into the target video, <Picture 1> (from [Shot 1]) is fully "\n            "referenced."\n        )\n    if task == "fl2va":\n        return (\n            "How the reference pictures align with the target video — Picture 1 (from Shot 1) aligns with the "\n            f"0.00-second mark of the target video; Picture 2 (from Shot {final_shot}) aligns with the "\n            f"{duration:.2f}-second mark of the target video."\n        )\n    if task == "l2va":\n        return (\n            "How the reference pictures align with the target video — <Picture 1> (from [Shot "\n            f"{final_shot}]) aligns with the {duration:.2f}-second mark of the target video."\n        )\n    raise ValueError(f"Unknown base task {task!r}.")\n\n\ndef reference_labels(references) -> list[tuple[str, str]]:\n    """The label MiniMax-H3\'s `ref2va` presentation gives every reference, in packed order.\n\n    Mirrors `tokenize_ref2va_presentation`: labels are numbered *per modality*, and a video that carries a soundtrack\n    is labelled `<Audio j>` before `<Video k>`, so a rewrite that cites the wrong number cites the wrong asset.\n\n    Args:\n        references (`list[MiniMaxH3Reference]`): The references of the request, in packed order.\n\n    Returns:\n        `list[tuple[str, str]]`: `(kind, label)` per reference, where a sound-bearing video\'s label names both.\n    """\n    counts = {"image": 0, "video": 0, "audio": 0}\n    labels = []\n    for reference in references:\n        audio_label = None\n        if reference.has_audio:\n            counts["audio"] += 1\n            audio_label = f"<Audio {counts[\'audio\']}>"\n        if reference.kind == "image":\n            counts["image"] += 1\n            labels.append(("image", f"<Picture {counts[\'image\']}>"))\n        elif reference.kind == "video":\n            counts["video"] += 1\n            label = f"<Video {counts[\'video\']}>"\n            labels.append(("video", f"{audio_label} + {label}" if audio_label else label))\n        else:\n            labels.append(("audio", audio_label))\n    return labels\n\n\ndef reference_duration_seconds(references, num_frames) -> float | None:\n    """The duration a `ref2va` request generates, including the case where the soundtrack sets it.\n\n    `num_frames` may be left open, and the `ref2va` setup step then takes the duration off the single audio-bearing\n    reference. The rewrite needs that number to place its cut timestamps, so the same resolution is repeated here —\n    on the already-decoded waveform, so it costs nothing — and simply gives up (`None`) if the request is one setup\n    would reject anyway.\n    """\n    if num_frames:\n        return duration_seconds(num_frames)\n    audio_bearing = [reference for reference in references if reference.has_audio]\n    if len(audio_bearing) != 1:\n        return None\n    reference = audio_bearing[0]\n    # A decoded reference always carries the rate its container reported; one built from an in-memory waveform without\n    # a rate is resolved against the audio VAE\'s own, which is the pipeline\'s to know, so the hint is left out.\n    if not reference.sample_rate:\n        return None\n    seconds = reference.audio.shape[-1] / reference.sample_rate\n    return duration_seconds(round(seconds * MINIMAX_H3_FPS))\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# The system prompts: the two official guides, condensed to what a rewrite has to obey.\n# ---------------------------------------------------------------------------------------------------------------\n\n_SHARED_RULES = """\\\n## Shots and cuts\n\nBegin the body with `[Shot 1]`, which never carries a timestamp. Later shots open with a strictly increasing cut time\nthat falls inside the video duration, written `[Shot 2] At 00:03.500, the camera cuts to ...` (`MM:SS.mmm`). Use `the\ncamera cuts to`, `the shot cuts to`, `the shot transitions to`, `the shot changes to` or `the shot switches to` for an\nordinary cut; cross-dissolve, fade or wipe only when the user asked for one. A cut must introduce new information\nabout the subject, space, state, viewpoint or time — if only the distance or the angle changes, move the camera\ninstead. Never place a cut at or past the end of the video.\n\nA cut is expressed *only* by opening the next `[Shot N]` with its timestamp. Never describe a transition, a cut or a\njump to another scene in the middle of a shot: if the picture changes, the shot has ended and the next `[Shot N]`\nbegins. Everything inside one `[Shot N]` is one continuous take.\n\n## Camera motion\n\nWrite camera motion as natural English inside the shot, never as labels stacked at the end of a sentence. Motion type\nis one of `Zoom In / Zoom Out`, `Push In / Pull Out`, `Pan Left / Pan Right`, `Truck Left / Truck Right`, `Tilt Up /\nTilt Down`, `Pedestal Up / Pedestal Down`, `Arc Shot`, `Tracking Shot`, `Static Shot`, `Shake Slightly / Shake\nStrongly`, `POV`, `Roll Clockwise / Roll Counterclockwise`. Add amplitude (`with small amplitude`, `with large\namplitude`) and speed (`at slow speed`, `at fast speed`) only when they mean something. Medium amplitude and normal\nspeed are the default and are left unsaid — never write `with medium amplitude`, `at normal speed` or any other\namplitude or speed phrase than the four above.\n\n    The camera pushes in with small amplitude at slow speed toward the folded letter in her hands.\n    The camera holds a static shot as the runner exits the frame.\n\n## Speakers, dialogue and singing\n\nAnyone who speaks, sings or produces an off-screen human voice gets a stable ID: `(S1)`, `(S2)`, and `(S1,S2)` when\nalready-numbered speakers vocalize together. An ID belongs to the same person across every shot; characters who never\nvocalize get none. At a speaker\'s first appearance, establish a stable identity from what is visible and audible —\ncharacter type, age, gender, on-screen or not, pitch, timbre, speaking rate, accent.\n\nThe identifying phrase, the ID, the action and the delivery go *outside* `<d>`. Inside `<d>` there is only a language\ntag and the spoken content itself, every original word and punctuation mark preserved verbatim, never translated and\nnever rewritten. The wrapper is literally the two characters-plus-one letter `<d>` and its closing `</d>` — never any\nother tag name, and never a tag out of your own chat vocabulary:\n\n    The young woman with a quiet, breathy voice (S1) says: <d>[English] I get off at the next station.</d>\n    The two children (S1,S2) shout together, <d>[English] Wait for us!</d>\n\nFor voiceover use the exact phrase `says in an off-screen voiceover`, and immediately after that `<d>` block state\nthat the on-screen character\'s lips remain closed. When one line crosses a cut, write `<scenetrans>` at the\nconnecting point in both parts and say the audio continues across the cut (`continues seamlessly across the cut`,\n`continues uninterrupted into the next shot`, `carries over from the previous shot`, `remains audible across the\ntransition`). Use `<cutoff>` when speech is truncated by the end of the video.\n\nNever invent dialogue the user did not ask for. If the user gave no spoken lines, write no `<d>` blocks at all.\n\n## On-screen text\n\nAny banner, sign, label, subtitle or neon text actually visible on screen goes in English double quotation marks,\nverbatim and untranslated: `A red neon sign reading "营业中" glows above the doorway.`\n\n## overall_soundscape\n\nOne continuous paragraph, 1-4 English sentences, summarizing ambience, physical action sounds and non-verbal human\nsounds across the whole video: wind, rain, traffic, footsteps, fabric, impacts, breathing, laughter, panting.\nDialogue, singing and diegetic music belong in the description body and are not repeated here. `N/A` only when the\nuser explicitly asked for complete silence.\n\n## non_diegetic_music\n\n1-3 English sentences on the background music only the audience hears. Instrumentation, tempo, rhythm and dynamic\nchange — no abstract mood words and no explanation of what the score does emotionally. Music the characters can hear\n(singing, instruments, a radio, a television, a phone) is diegetic and belongs in the body instead. `N/A` when there\nis no non-diegetic music.\\\n"""\n\nBASE_SYSTEM_PROMPT = f"""\\\nYou are H3-Context-IR, the prompt rewriter of the MiniMax-H3 audiovisual generation model. You turn a user\'s raw\nrequest into the structured representation H3 was trained on. Everything you write is consumed directly by the\ngenerator, so you never address the user and never explain yourself.\n\n# Output contract\n\nEmit exactly these three fields, in this order, each separated from the next by one blank line, and nothing else:\n\n    integrated_multimodal_description: [Shot 1] ...\n\n    overall_soundscape: ...\n\n    non_diegetic_music: ...\n\nHard rules:\n\n- Plain text only. No markdown, no code fences, no headings, no bullet lists, no preamble, no closing remark.\n- Start your very first token with `integrated_multimodal_description:`.\n- Never write the picture-alignment instruction line (`For the target video, ...` / `How the reference pictures align\n  ...`). The system prepends it. Writing one yourself is an error.\n- Write in English. Only dialogue and lyrics inside `<d>`, and text visibly present in the scene, keep their original\n  language.\n- Stay faithful to the user\'s request: keep every subject, action, line of dialogue and sound they specified. Add\n  scene, character, action and sound detail only where it is consistent with their intent.\n\n# integrated_multimodal_description\n\nThis is the body of the rewrite, developed along the timeline. Every detail must correspond to something that can be\nseen or heard: visual style, opening composition, subject appearance and position, scene and key props, actions and\nreactions, cuts, spoken language, and sound synchronized to what happens.\n\nOpen `[Shot 1]` with the overall style and the initial composition — `Cinematic`, `live-action`, `2D-animated`,\n`3D CG`, `claymation`, `watercolor`, `vintage film` and the like. With keyframes, read the style off the reference\nimage; without them, take it from the user\'s text.\n\n    [Shot 1] Live-action, cinematic, a medium-wide shot frames...\n\n{_SHARED_RULES}\n\n# Working from keyframes\n\n**i2va (first frame given).** `<Picture 1>` *is* the frame at 0.00 seconds and belongs to `[Shot 1]`. Establish the\nstyle, subjects, composition and scene anchors of the image first, then the action that follows. Character identity,\nclothing, colors, key objects and spatial relationships stay consistent. Shape: first-frame anchor, action onset,\ncontinuous development, result or reaction.\n\n**fl2va (first and last frame given).** Picture 1 opens and Picture 2 ends. Do not describe two static images — supply\nthe motion path between them: how the subject moves, how the pose changes, how objects are handled, how the\ncomposition, scene and lighting evolve. Prefer a single shot so the model can interpolate continuously, unless the\nuser explicitly asked for more, and reach Picture 2\'s state at the very end of the final shot. Shape: first-frame\nstate, observable intermediate changes, progressively narrowing differences, last-frame state.\n\n**l2va (last frame given).** `<Picture 1>` is the *final* frame and belongs to the last shot, not to Shot 1. Infer a\nplausible earlier state from the user\'s intent and the image, then describe how characters, objects, camera and scene\ngradually approach it. Shape: plausible preceding state, explicit action and transition path, gradual convergence,\nlast-frame landing.\n\n# Example (text-only request, 8.00 seconds)\n\nintegrated_multimodal_description: [Shot 1] Live-action, cinematic, a medium-wide shot frames a baker opening the\nshutters of a small street bakery before sunrise. The camera pushes in with small amplitude at slow speed as the\nmiddle-aged baker with a calm, slightly raspy voice (S1) places a fresh loaf on the wooden counter and says:\n<d>[English] First batch of the morning.</d> [Shot 2] At 00:05.000, the camera cuts to a close-up of steam rising\nfrom the sliced bread while the baker\'s final words carry over from the previous shot.\n\noverall_soundscape: Wooden shutters scrape open over a quiet street as trays clink softly inside the bakery. The\ndoorbell rings once, followed by light footsteps and the crisp sound of bread being sliced.\n\nnon_diegetic_music: A soft acoustic-guitar pattern at a moderate tempo, joined by sparse upright-bass notes and a\ngentle fade at the end.\n\n# Example (first frame given, 8.00 seconds)\n\nintegrated_multimodal_description: [Shot 1] Live-action, cinematic, the young woman shown in <Picture 1> remains\nbeside the rain-covered train window, preserving her appearance, clothing, seat position, and the carriage layout.\nThe camera trucks right with small amplitude at slow speed as she lifts her gaze from the folded letter toward the\npassing city lights. Her reflection moves across the glass while the quiet, breathy young woman (S1) says:\n<d>[English] I get off at the next station.</d> She folds the letter along its existing crease.\n\noverall_soundscape: The train wheels produce a steady metallic rhythm beneath a low ventilation hum. Rain ticks\nagainst the window while paper rustles softly in her hands.\n\nnon_diegetic_music: Sustained cello notes at a slow tempo with widely spaced piano tones, gradually decreasing in\nvolume.\\\n"""\n\nREF_SYSTEM_PROMPT = f"""\\\nYou are H3-Context-IR, the prompt rewriter of the MiniMax-H3 audiovisual generation model, working in\n**full-reference mode**: the request comes with reference images, videos and audio, and your rewrite has to state\nwhat each of them contributes to the target video. Everything you write is consumed directly by the generator, so you\nnever address the user and never explain yourself.\n\n# Output contract\n\nEmit exactly these six sections, in this order, each section name on its own line followed by its content, and one\nblank line between sections. Nothing else.\n\n    subject_definitions:\n    <one line per referenced item>\n\n    summary:\n    [task type] one short paragraph\n\n    retention_analysis:\n    <one line per reference label>\n\n    detailed_description:\n    <one or two sentences of style, then the shots>\n\n    overall_soundscape:\n    ...\n\n    non_diegetic_music:\n    ...\n\nHard rules:\n\n- Plain text only. No markdown, no code fences, no bullet lists, no preamble, no closing remark.\n- Start your very first token with `subject_definitions:`.\n- Write in English. Only dialogue and lyrics inside `<d>`, and text visibly present in the scene, keep their original\n  language.\n- Use only the reference labels the request lists, exactly as spelled there. Never invent a label, never renumber one,\n  and never introduce a new label after `subject_definitions`.\n- A label keeps one meaning across all six sections.\n\n# subject_definitions\n\nOne line per piece of referenced content that is tracked separately later. Four label types:\n\n- `<Subject N>` — visible content reused or modified in the target video: people, animals, objects, scenes,\n  backgrounds, environments, clothing, props, interfaces, effects, styles, actions, expressions, poses. It is a\n  *content unit*, not a file. One subject may come from several assets, and one asset may provide several subjects.\n- `<Picture N>` — a reference image used as a concrete frame anchor (first frame, keyframe, last frame, edited\n  keyframe) or as a storyboard / composition reference.\n- `<Video N>` — a reference video in a whole-video relationship: the source being edited, the clip being continued,\n  or a reference for camera movement, cuts, rhythm or temporal structure.\n- `<Audio N>` — an audio asset, or a reference video\'s synchronized track, that is copied or referenced.\n\nSay what the label denotes, what role it plays and which features to follow; name the source asset when provenance\nwould otherwise be ambiguous.\n\n    <Subject 1> is the young woman in <Picture 1>, with long dark hair, a blue cardigan, and a thin silver necklace.\n    <Subject 1> is the woman whose appearance comes from <Picture 1> and whose walking motion comes from <Video 1>.\n    <Picture 2> is the first frame of [Shot 1], showing a woman seated beside a café window.\n    <Video 1> is the source video for the target video edit.\n    <Audio 1> is the voice-timbre reference for <Subject 1> (S1).\n\nA label is only ever for content that is actually *in* one of the reference assets listed for this request. Anything\nthat exists only in the user\'s text — a character, a place, a weather condition nobody sent you a picture of — is\nwritten into `detailed_description` as ordinary description and gets no `<Subject N>` of its own. Never define a\nsubject you cannot see in a reference.\n\nAn image used *only* to define a character, scene, costume or style gets no standalone `<Picture N>` line — cite it\ninside the `<Subject N>` definition it supports. A person, object, scene, action or effect taken out of a reference\nvideo is a `<Subject N>`, not a `<Video N>`. A reference video does not produce an `<Audio N>` merely because the file\nhas sound. When an `<Audio N>` corresponds to a target speaker, reuse that speaker\'s global ID: `<Subject N> (Sx)`,\nor a stable voice description followed by `(Sx)`.\n\n# summary\n\nOne short English paragraph, opening with a square-bracketed task-type prefix. Task types:\n\n| `keyframe completion` | an image is a concrete frame anchor of the target video |\n| `reference generation` | an image, video or audio guides generation of a character, scene, style, action, camera\n  movement or storyboard, without being a frame anchor or the video being edited or continued |\n| `video editing` | an existing source video is directly modified |\n| `video continuation` | new content continues, extends, resumes or transitions from a source video |\n| `audio reuse` | the same audio signal is reused in full or in part |\n| `audio reference` | only the music style, timbre, dialogue or lyric content, sound texture, beat or continuity of\n  an audio signal is referenced, not the signal itself |\n\nCombine several with ` + `, never repeating one: `[video continuation + keyframe completion]`, `[video editing +\naudio reuse]`. The mere presence of a video or an audio file does not create a task type — a video that only lends\ncamera movement, cuts or rhythm is `reference generation`. For a video-editing task, the paragraph begins right after\nthe prefix with `The target video is an edited version of <Video 1>.`\n\n# retention_analysis\n\nOne line per reference label, in the meaning `subject_definitions` gave it. Visible content\n(`<Subject N>`, `<Picture N>`, `<Video N>`) uses `fully_preserved`, `partially_preserved`, `attribute_transfer` or\n`weak_reference`. Audio (`<Audio N>`) uses `fully_copy`, `partially_copy`, `reference` or `weak_reference`. These\nmarkers are fixed values — write them exactly.\n\n    <Subject 1> (appears in [Shot 1], [Shot 3]): fully_preserved - ...\n    <Picture 2> ([Shot 1] first frame): fully_preserved - ...\n    <Video 1> (cut and pacing structure): weak_reference - ...\n    <Audio 1>: fully_copy - <Audio 1> is reused 1:1 as the target video\'s complete final audio track.\n\nStay inside the role the label was already given. New actions, backgrounds or plot events in the target video are not\nlosses of fidelity. Never write `(Sx)` in this section.\n\n# detailed_description\n\nThe main body, shot by shot in playback order, with reference labels inserted where they apply. Establish the style\nin one or two English sentences *before* `[Shot 1]` — unlike text-only mode, where the style opens Shot 1.\n\nMake this as detailed and explicit as you can. For every shot: the current composition, subject appearance and\nposition, environment and lighting, actions and state changes, camera movement, the sound at that moment, and the\npoint where referenced content actually appears or takes effect. Never reduce it to a plot summary or a list of\nreference relationships. Aim for 350-500 English words for a generation task; dialogue-heavy content prioritizes\nfitting the whole spoken timeline over any word count. A single shot is not a licence to write less.\n\nAt an important `<Subject N>`\'s first clear appearance, describe its referenced characteristics, where it sits in the\nframe and what it is doing — then keep using the label without redefining it. Frame anchors read naturally: `the shot\nbegins from <Picture 1>`, `the shot\'s keyframe corresponds to <Picture 2>`, `the shot ends on <Picture 3>`. Cite\n`<Video N>` where its source state, structure or continuation relationship applies, and `<Audio N>` in the shot or\nphase where its audio relationship is active.\n\nWhen a referenced subject speaks, keep both labels: `<Subject 2> (S1) turns toward the woman and says, <d>[English]\nLast summer, I went to my grandfather\'s house.</d>`. `<Subject N>` is the referenced subject, `(Sx)` the actual\nspeaker; off-screen keeps the same form and is marked `off-screen`. A speaker with no defined subject gets a stable\nvoice description plus `(Sx)`. Verbal content that exists only inside a directly reused soundtrack uses `<Audio N>`\nas its source and gets no `(Sx)`. Assign `(Sx)` once, in the order vocal events happen in the target video.\n\nWhen dialogue, narration or lyrics are reused from reference audio, or the user asked for them to be reperformed,\nkeep the exact source words and original language inside `<d>`, write `[unclear]` for spans you cannot make out, and\nstandardize punctuation to `,` `.` `?` `!` — no tildes, emoji, bullets or decorative repeats. When only timbre,\nrhythm, emotion or delivery is referenced, do not carry the original words over.\n\n{_SHARED_RULES}\n\nState an audio copy-or-reference relationship in the section that matches the audible layer: ambience and effects in\n`overall_soundscape`, audience-only score in `non_diegetic_music`. Complete dialogue and lyrics appear only inside\n`<d>` in `detailed_description`.\n\n# Example\n\nsubject_definitions:\n<Subject 1> is the coffee-shop environment in <Picture 1>, featuring an exposed brick wall, an orange tufted sofa with\npatterned pillows, a neon sign, and a wooden coffee table.\n<Subject 2> is the fluffy white Samoyed in <Picture 2>, with thick white fur, pointed ears, a dark nose, and a curved\ntail.\n<Subject 3> is the young blonde woman in <Video 1>, with long blonde hair and a light-pink button-down shirt with\nrolled-up sleeves.\n<Audio 1> is the voice-timbre reference for <Subject 3> (S1), containing a spoken English vocal layer.\n\nsummary:\n[reference generation + audio reference] The target video shows <Subject 3> eating a cookie in <Subject 1>. A young man\nenters with <Subject 2>, which lunges toward the cookie. The two-shot exchange uses <Audio 1> as the voice-timbre\nreference for <Subject 3> and ends with a canned audience laugh.\n\nretention_analysis:\n<Subject 1> (appears in [Shot 1], [Shot 2]): fully_preserved - the exposed brick wall, orange tufted sofa, patterned\npillows, neon sign, and wooden coffee table are retained.\n<Subject 2> (appears in [Shot 1], [Shot 2]): fully_preserved - the Samoyed\'s thick white fur, pointed ears, dark nose,\nand curved tail are retained.\n<Subject 3> (appears in [Shot 1], [Shot 2]): fully_preserved - the blonde woman\'s identity, long hair, and light-pink\nshirt are retained.\n<Audio 1>: reference - its vocal timbre guides the dialogue delivery of <Subject 3> without copying the original\nsignal.\n\ndetailed_description:\nThe target video uses a realistic multi-camera sitcom style with warm indoor lighting.\n[Shot 1] A medium shot establishes <Subject 1>, the coffee shop with its exposed brick wall, orange tufted sofa,\npatterned pillows, neon sign, and wooden coffee table. <Subject 3> (S1), the young woman with long blonde hair and a\nlight-pink button-down shirt with rolled-up sleeves, sits on the sofa holding a chocolate-chip cookie. From the left, a\nyoung man in a dark-grey hoodie enters holding the leash of <Subject 2>, the thick-furred white Samoyed with pointed\nears, a dark nose, and a curved tail. The dog lunges toward the cookie and pulls the leash taut. <Subject 3> (S1) jerks\nher hand back and, using the clear youthful voice timbre referenced from <Audio 1>, exclaims with light annoyance,\n<d>[English] Hey! Watch your dog!</d> She closes her lips and guards the cookie while the man pulls the dog back.\n[Shot 2] At 00:03.000, the shot cuts to a close-up of <Subject 3> (S1), the blonde woman in the light-pink shirt from\nShot 1. Her annoyance softens as she looks toward the Samoyed. The camera pushes in with small amplitude at slow speed\nwhile a classic canned audience laugh begins and continues through the final frame.\n\noverall_soundscape:\nSoft indoor coffee-shop room tone continues throughout the scene, with the leash snapping taut and the dog\'s claws\nscrabbling on the wooden floor.\n\nnon_diegetic_music:\nN/A\\\n"""\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# The request the rewriter is shown.\n# ---------------------------------------------------------------------------------------------------------------\n\n_BASE_TASK_BRIEF = {\n    "t2va": "t2va — text only, no reference image. Build the whole timeline from the request.",\n    "i2va": (\n        "i2va — <Picture 1> is the first frame of the video, at 0.00 seconds, and belongs to [Shot 1]. Cite "\n        "<Picture 1> explicitly in [Shot 1] and anchor its style, subjects and composition before any action."\n    ),\n    "fl2va": (\n        "fl2va — Picture 1 is the first frame, at 0.00 seconds, and Picture 2 is the last frame, at the end of the "\n        "video. Write the motion path between them, and prefer a single shot."\n    ),\n    "l2va": "l2va — <Picture 1> is the LAST frame of the video and belongs to the final shot, not to [Shot 1].",\n}\n\n\ndef base_user_message(prompt: str, task: str, duration: float, keyframes: list[Image.Image]) -> list[dict]:\n    """The chat turn a base-task rewrite is asked for: the keyframes under their labels, then the request."""\n    content: list[dict] = []\n    labels = ["<Picture 1>", "<Picture 2>"]\n    for index, keyframe in enumerate(keyframes):\n        content.append({"type": "text", "text": f"{labels[index]}:"})\n        content.append({"type": "image", "image": _thumbnail(keyframe)})\n    content.append(\n        {\n            "type": "text",\n            "text": (\n                f"Task: {_BASE_TASK_BRIEF[task]}\\n"\n                f"Video duration: {duration:.2f} seconds at 24 fps. Every cut timestamp must fall strictly inside it.\\n"\n                f"\\nUser request:\\n{prompt.strip()}\\n"\n                "\\nAll three fields are mandatory, in order — write `non_diegetic_music: N/A` if the video carries no "\n                "audience-only score. Write the rewrite now, starting with `integrated_multimodal_description:`."\n            ),\n        }\n    )\n    return [{"role": "user", "content": content}]\n\n\ndef reference_user_message(prompt: str, labels, references, duration: float | None) -> list[dict]:\n    """The chat turn a `ref2va` rewrite is asked for: every reference under its own label, then the request.\n\n    An audio reference never reaches the rewriter — Qwen3-VL has no audio tower, and it never reaches the conditioner\n    either — so it is announced by label and role only, and the rewrite has to place its relationship without having\n    heard it. A video is shown the way the conditioner sees it: sampled down to 2 fps, from the frames the reference\n    already decoded.\n    """\n    content: list[dict] = []\n    inventory = []\n    for (kind, label), reference in zip(labels, references):\n        inventory.append(f"{label} — {kind} reference")\n        if kind == "image":\n            content.append({"type": "text", "text": f"{label}:"})\n            content.append({"type": "image", "image": _thumbnail(reference.image)})\n        elif kind == "video":\n            content.append({"type": "text", "text": f"{label}:"})\n            content.append({"type": "video", "video": _video_frames(reference)})\n        else:\n            content.append(\n                {"type": "text", "text": f"{label}: an audio reference. You cannot hear it; describe its role only."}\n            )\n\n    duration_line = (\n        f"Video duration: {duration:.2f} seconds at 24 fps. Every cut timestamp must fall strictly inside it.\\n"\n        if duration is not None\n        else "Video duration: taken from the reference soundtrack. Keep cut timestamps early and conservative.\\n"\n    )\n    content.append(\n        {\n            "type": "text",\n            "text": (\n                "Task: full-reference generation (ref2va).\\n"\n                "References, in the order the generator reads them, under the labels you must use:\\n"\n                + "\\n".join(f"  {entry}" for entry in inventory)\n                + "\\n"\n                + duration_line\n                + f"\\nUser request:\\n{prompt.strip()}\\n"\n                "\\nAll six sections are mandatory, in order. Write the rewrite now, starting with "\n                "`subject_definitions:`."\n            ),\n        }\n    )\n    return [{"role": "user", "content": content}]\n\n\ndef _as_uint8_rgb(media) -> np.ndarray:\n    """An in-memory reference frame (or frame stack) as channels-last `uint8` RGB.\n\n    The library used to expose this as `reference_media_to_uint8`; the reference dataclasses replaced it, and the one\n    normalization left to do here is for media a caller built in memory rather than through `from_file` — which is\n    already `uint8` `(num_frames, height, width, 3)`. A `torch.Tensor` is channels-first, as everywhere else in\n    diffusers, a `np.ndarray` channels-last, and floating point values are read over `[0, 1]`.\n    """\n    if isinstance(media, list):\n        return np.stack([_as_uint8_rgb(item) for item in media])\n    if isinstance(media, Image.Image):\n        return np.asarray(media.convert("RGB"))\n    if isinstance(media, torch.Tensor):\n        media = media.movedim(-3, -1).cpu().numpy()\n    media = np.asarray(media)\n    if media.dtype != np.uint8:\n        media = (media * 255.0).round().clip(0, 255).astype(np.uint8)\n    return media\n\n\ndef _thumbnail(image) -> Image.Image:\n    """A copy of an image small enough to look at cheaply. The encoding path never sees it."""\n    if not isinstance(image, Image.Image):\n        image = Image.fromarray(_as_uint8_rgb(image))\n    image = image.convert("RGB")\n    if max(image.size) > REWRITE_IMAGE_LONG_EDGE:\n        scale = REWRITE_IMAGE_LONG_EDGE / max(image.size)\n        image = image.resize((max(1, round(image.width * scale)), max(1, round(image.height * scale))), Image.LANCZOS)\n    return image\n\n\ndef _video_frames(reference) -> list[Image.Image]:\n    """A reference video at the 2 fps the conditioner reads it, capped so one clip cannot own the prefill.\n\n    Sampled off the frames the reference already decoded rather than off the file, so the rewriter looks at the same\n    footage the conditioner will and no container is opened twice.\n    """\n    frames = _as_uint8_rgb(reference.frames)\n    stride = max(1, round((reference.fps or MINIMAX_H3_FPS) / REWRITE_VIDEO_FPS))\n    sampled = list(range(0, len(frames), stride))[:REWRITE_VIDEO_MAX_FRAMES]\n    return [_thumbnail(Image.fromarray(frames[index])) for index in sampled]\n\n\n# ---------------------------------------------------------------------------------------------------------------\n# Generation and cleanup.\n# ---------------------------------------------------------------------------------------------------------------\n\nBASE_FIELDS = ("integrated_multimodal_description", "overall_soundscape", "non_diegetic_music")\nREF_FIELDS = (\n    "subject_definitions",\n    "summary",\n    "retention_analysis",\n    "detailed_description",\n    "overall_soundscape",\n    "non_diegetic_music",\n)\n\n_FENCE = re.compile(r"^\\s*```[a-zA-Z]*\\s*|\\s*```\\s*$")\n# The shapes a chat model signs off with. Only ever matched against *trailing* lines, so a rewrite that happens to\n# contain one of these phrases inside a shot description keeps it.\n_CHATTER = re.compile(\n    r"^\\s*(?:```.*|(?:let me know|i hope|hope (?:this|that)|would you like|note|if you (?:want|need)|feel free|"\n    r"this (?:rewrite|prompt|follows))\\b.*)$",\n    re.IGNORECASE,\n)\n_INSTRUCTION_LINE = re.compile(\n    r"^\\s*(For the target video,.*?referenced\\.|How the reference pictures align with the target video[^\\n]*)\\s*",\n    re.IGNORECASE,\n)\n\n\ndef _generate(components, messages: list[dict], system: str, max_new_tokens: int) -> tuple[str, bool]:\n    """One greedy decode off the conditioner\'s own language-model head.\n\n    The conditioner is the full `Qwen3VLForConditionalGeneration`, so `generate` is available as it stands: the\n    encoding path just never calls it, reading `hidden_states[50]` off `text_encoder.model` instead. Greedy, because a\n    rewrite is a format-following task and sampling only adds ways to leave the format.\n    """\n    import torch\n\n    model, processor = components.text_encoder, components.processor\n    device = getattr(components, "_execution_device", None) or next(model.parameters()).device\n\n    started = time.time()\n    inputs = processor.apply_chat_template(\n        [{"role": "system", "content": system}, *messages],\n        tokenize=True,\n        add_generation_prompt=True,\n        return_dict=True,\n        return_tensors="pt",\n        # A frame list is already at the rate it should be read at, so no resampling in the processor.\n        processor_kwargs={"do_sample_frames": False},\n    )\n    inputs = {name: value.to(device) if hasattr(value, "to") else value for name, value in inputs.items()}\n    prefill = int(inputs["input_ids"].shape[1])\n\n    with torch.inference_mode():\n        generated = model.generate(\n            **inputs,\n            max_new_tokens=max_new_tokens,\n            do_sample=False,\n            num_beams=1,\n            repetition_penalty=1.0,\n        )\n    completion = generated[0, prefill:]\n    text = processor.batch_decode([completion], skip_special_tokens=True)[0]\n    # A decode that used its whole budget stopped where the budget ran out, not where the rewrite ended, and is the\n    # only case whose tail needs trimming. One that stopped on its own end token is complete by construction.\n    truncated = int(completion.shape[0]) >= max_new_tokens\n    print(\n        f"[rewrite] {prefill} prefill + {int(completion.shape[0])} decoded tokens in {time.time() - started:.1f}s"\n        f"{\' (hit the budget)\' if truncated else \'\'}",\n        flush=True,\n    )\n    if os.environ.get("REWRITE_DEBUG"):\n        print(f"[rewrite] raw:\\n{text}\\n[rewrite] end raw", flush=True)\n    return text, truncated\n\n\ndef _drop_incomplete_tail(text: str, fields: tuple[str, ...]) -> str:\n    """Cut a decode that ran into the token budget back to something well formed.\n\n    A rewrite that stops mid-sentence is still fed to the encoder, so the tail is trimmed to the last sentence that\n    actually ended — and a whole trailing field that never got past its own name is dropped.\n    """\n    blocks = _split_fields(text, fields)\n    if not blocks:\n        return text\n    name, body = blocks[-1]\n    if not body.strip():\n        blocks = blocks[:-1]\n    elif body.rstrip()[-1] not in ".!?\\"\'>":\n        cut = max(body.rfind(mark) for mark in (". ", ".\\n", "! ", "!\\n", "? ", "?\\n", "</d>"))\n        if cut > 0:\n            end = cut + (4 if body[cut : cut + 4] == "</d>" else 1)\n            blocks[-1] = (name, body[:end])\n        else:\n            blocks = blocks[:-1]\n    if not blocks:\n        return text\n    return "\\n\\n".join(f"{name}:{body}" if body.startswith("\\n") else f"{name}: {body.strip()}" for name, body in blocks)\n\n\ndef _split_fields(text: str, fields: tuple[str, ...]) -> list[tuple[str, str]]:\n    """The `name: body` blocks of a rewrite, in the order they appear."""\n    pattern = re.compile(rf"^({\'|\'.join(fields)})\\s*:", re.MULTILINE)\n    matches = list(pattern.finditer(text))\n    return [\n        (match.group(1), text[match.end() : matches[index + 1].start() if index + 1 < len(matches) else len(text)])\n        for index, match in enumerate(matches)\n    ]\n\n\ndef _strip_trailing_chatter(text: str) -> str:\n    """Drop the sign-off a chat model puts after the body: a closing fence, an offer to revise, a note."""\n    lines = text.rstrip().split("\\n")\n    while lines and (not lines[-1].strip() or _CHATTER.match(lines[-1])):\n        lines.pop()\n    return "\\n".join(lines)\n\n\ndef _scrub_default_camera_qualifiers(text: str) -> str:\n    """Delete the amplitude and speed phrases the guide says are the unsaid defaults.\n\n    `with medium amplitude` and `at normal speed` are exactly what "medium amplitude and normal speed are usually\n    omitted" rules out, and the model reaches for them anyway. They carry no information, so removing them is a pure\n    deterministic normalization rather than a judgement about the rewrite.\n    """\n    text = re.sub(r"\\s+(?:with|at)\\s+(?:medium|moderate)\\s+amplitude", "", text, flags=re.IGNORECASE)\n    return re.sub(r"\\s+at\\s+(?:normal|medium|moderate|steady)\\s+speed", "", text, flags=re.IGNORECASE)\n\n\ndef _repair_dialogue_tags(text: str) -> str:\n    """Put a spoken line back inside `<d>` when the model reached for a tag out of its own chat vocabulary.\n\n    Qwen3-VL occasionally writes `<tool_call>[English] ...</tool_call>` where the format wants\n    `<d>[English] ...</d>` — the language tag and the line itself are right, only the wrapper is the model\'s own. The\n    wrapper is what MiniMax-H3 keys dialogue off, so it is renamed rather than left to be encoded as prose.\n    """\n    for tag in set(re.findall(r"<([a-zA-Z_][\\w-]*)>\\s*\\[[A-Z]", text)) - {"d"}:\n        text = text.replace(f"<{tag}>", "<d>").replace(f"</{tag}>", "</d>")\n    return text\n\n\ndef clean_core(text: str, fields: tuple[str, ...], truncated: bool = False) -> str:\n    """Strip everything a chat model adds around the body it was asked for.\n\n    Fences, an echoed instruction line, a leading `assistant`, a stray closing remark after the last field, a dialogue\n    wrapper out of the model\'s own chat vocabulary, and — only when the decode ran out of budget — a tail that stops\n    mid-sentence. What is left is the prompt body itself, which is what the encoder is handed.\n    """\n    text = text.strip()\n    text = _FENCE.sub("", text).strip()\n    text = _strip_trailing_chatter(text)\n    text = re.sub(r"^assistant\\s*[:\\n]", "", text, flags=re.IGNORECASE).strip()\n    # The instruction line is the caller\'s to assemble, so an echoed one is dropped rather than duplicated.\n    text = _INSTRUCTION_LINE.sub("", text).strip()\n    text = _repair_dialogue_tags(text)\n    text = _scrub_default_camera_qualifiers(text)\n\n    blocks = _split_fields(text, fields)\n    if blocks:\n        # Anything before the first field name is chatter; anything after the last field\'s body is a closing remark.\n        text = text[text.index(blocks[0][0]) :].strip()\n    elif "[Shot 1]" in text and fields is BASE_FIELDS:\n        # The body without its field name still is a body; name it rather than throwing the rewrite away.\n        text = f"{BASE_FIELDS[0]}: {text}"\n\n    if truncated:\n        text = _drop_incomplete_tail(text, fields)\n    # One blank line between fields, whatever the model emitted.\n    text = re.sub(r"\\n{3,}", "\\n\\n", text)\n    for name in fields[1:]:\n        text = re.sub(rf"\\n+({name}\\s*:)", r"\\n\\n\\1", text)\n    return text.strip()\n\n\ndef refine_keyframe_prompt(components, prompt, image, last_image, num_frames, max_new_tokens) -> str:\n    """Rewrite a `/encode` request into MiniMax-H3\'s trained prompt format.\n\n    Args:\n        components: The conditioner pipeline, for its `text_encoder` and `processor`.\n        prompt (`str`): The user\'s raw prompt.\n        image (`PIL.Image.Image`, *optional*): The first keyframe, if any.\n        last_image (`PIL.Image.Image`, *optional*): The last keyframe, if any.\n        num_frames (`int`): The requested frame count, before alignment.\n        max_new_tokens (`int`): The decode budget.\n\n    Returns:\n        `str`: the final prompt — the code-assembled instruction line, a blank line, and the model\'s core fields.\n    """\n    task = base_task(image, last_image)\n    duration = duration_seconds(num_frames)\n    keyframes = [keyframe for keyframe in (image, last_image) if keyframe is not None]\n\n    raw, truncated = _generate(\n        components, base_user_message(prompt, task, duration, keyframes), BASE_SYSTEM_PROMPT, max_new_tokens\n    )\n    core = clean_core(raw, BASE_FIELDS, truncated)\n    if not core:\n        raise ValueError("The rewrite came back empty.")\n\n    instruction = base_instruction_line(task, duration, final_shot_index(core))\n    return core if instruction is None else f"{instruction}\\n\\n{core}"\n\n\ndef refine_reference_prompt(components, prompt, references, num_frames, max_new_tokens) -> str:\n    """Rewrite an `/encode_ref2va` request into MiniMax-H3\'s full-reference format.\n\n    Full-reference mode has no alignment instruction line: the six sections are the whole prompt, and the reference\n    relationships that a keyframe task states in one templated sentence are what `subject_definitions`,\n    `summary` and `retention_analysis` carry instead.\n\n    Args:\n        components: The conditioner pipeline, for its `text_encoder` and `processor`.\n        prompt (`str`): The user\'s raw prompt.\n        references (`list[MiniMaxH3Reference]`): The built references, in packed order — their labels and their media.\n        num_frames (`int` or `None`): The requested frame count, `None` when the soundtrack sets it.\n        max_new_tokens (`int`): The decode budget.\n\n    Returns:\n        `str`: the final prompt, the six sections as the model wrote them.\n    """\n    labels = reference_labels(references)\n    duration = reference_duration_seconds(references, num_frames)\n\n    raw, truncated = _generate(\n        components, reference_user_message(prompt, labels, references, duration), REF_SYSTEM_PROMPT, max_new_tokens\n    )\n    core = clean_core(raw, REF_FIELDS, truncated)\n    if not core:\n        raise ValueError("The rewrite came back empty.")\n    return core\n', encoding='utf-8')
(appdir / 'h3_blocks.py').write_text('"""Blocos split do MiniMax-H3, adaptados ao diffusers 0.40.0.\n\nO `h3_split_blocks.py` do Space usa classes que nao existem em 0.40.0\n(`MiniMaxH3AutoResizeStep`, `MiniMaxH3AutoKeyframeVaeEncoderStep`, `_generation_outputs`).\nAqui as duas metades sao montadas com os *auto steps* reais de 0.40.0:\n\n  Conditioner = AutoBeforeEncode (Ref2VASetup + Resize) + AutoTextEncoder\n  Generator   = AutoBeforeEncode + AutoVaeEncoder + AutoDenoise (inclui o after-denoise) + Decode\n\nA "wire" entre as duas fases continua sendo `prompt_embeds` + `text_token_tags`.\n"""\n\nimport torch\nfrom diffusers.modular_pipelines.minimax_h3.modular_blocks_minimax_h3 import (\n    MiniMaxH3AutoBeforeEncodeStep,\n    MiniMaxH3AutoDenoiseStep,\n    MiniMaxH3AutoTextEncoderStep,\n    MiniMaxH3AutoVaeEncoderStep,\n    MiniMaxH3DecodeStep,\n)\nfrom diffusers.modular_pipelines.modular_pipeline import SequentialPipelineBlocks\nfrom diffusers.modular_pipelines.modular_pipeline_utils import OutputParam\n\n\ndef conditioner_outputs():\n    return [\n        OutputParam.template("prompt_embeds", description="O hidden state que o transformer le, (1, N, 5120)."),\n        OutputParam("text_token_tags", type_hint=torch.Tensor, description="Tag de modalidade por linha de prompt_embeds."),\n        OutputParam("height", type_hint=int, description="Altura resolvida do video, em pixels."),\n        OutputParam("width", type_hint=int, description="Largura resolvida do video, em pixels."),\n    ]\n\n\ndef generator_outputs():\n    return [\n        OutputParam.template("videos", description="O video gerado, uma lista de imagens."),\n        OutputParam("audio", type_hint=None, description="A trilha gerada."),\n        OutputParam("sampling_rate", type_hint=int, description="Sample rate da trilha, em Hz."),\n    ]\n\n\nclass MiniMaxH3ConditionerBlocks(SequentialPipelineBlocks):\n    """Metade condicionadora: keyframes no canvas + leitura do Qwen3-VL na camada 50."""\n\n    model_name = "minimax-h3"\n    block_classes = [MiniMaxH3AutoBeforeEncodeStep, MiniMaxH3AutoTextEncoderStep]\n    block_names = ["before_encode", "text_encoder"]\n\n    @property\n    def description(self):\n        return "Coloca as keyframes no canvas e codifica a apresentacao em prompt_embeds + text_token_tags."\n\n    @property\n    def outputs(self):\n        return conditioner_outputs()\n\n\nclass MiniMaxH3GeneratorBlocks(SequentialPipelineBlocks):\n    """Metade geradora: VAE das keyframes + denoise + decode, sem o text encoder."""\n\n    model_name = "minimax-h3"\n    block_classes = [\n        MiniMaxH3AutoBeforeEncodeStep,\n        MiniMaxH3AutoVaeEncoderStep,\n        MiniMaxH3AutoDenoiseStep,\n        MiniMaxH3DecodeStep,\n    ]\n    block_names = ["before_encode", "vae_encoder", "denoise", "decode"]\n\n    @property\n    def description(self):\n        return "Recebe prompt_embeds + text_token_tags e gera video e audio."\n\n    @property\n    def outputs(self):\n        return generator_outputs()\n\n\ndef arm_vae_hooks(pipe):\n    """Faz os hooks de offload dispararem tambem em `vae.encode` e `vae.decode`.\n\n    `enable_auto_cpu_offload` envolve o `forward`. Os blocos chamam `vae.encode(...)`\n    e `vae.decode(...)` direto. Em 0.40.0 os dois metodos ja carregam\n    `@apply_forward_hook`, entao aqui e apenas uma garantia: `pre_forward` e idempotente\n    (se o modulo ja esta no device, ele nao faz nada).\n    """\n    for name in ("vae", "audio_vae"):\n        module = getattr(pipe, name, None)\n        if module is None:\n            continue\n        for method_name in ("encode", "decode"):\n            inner = getattr(module, method_name, None)\n            if inner is None or getattr(inner, "_h3_armed", False):\n                continue\n\n            def armed(*args, _module=module, _inner=inner, **kwargs):\n                hook = getattr(_module, "_hf_hook", None)\n                if hook is not None and hasattr(hook, "pre_forward"):\n                    hook.pre_forward(_module)\n                return _inner(*args, **kwargs)\n\n            armed._h3_armed = True\n            setattr(module, method_name, armed)\n', encoding='utf-8')
(appdir / 'h3_lora.py').write_text('"""Turbo LoRA do MiniMax-H3: carrega os dois LoRAs e dobra o ativo dentro dos pesos.\n\nAs duas metades do Space (`larry` na arvore de referencia, `lightx` na arvore do\ndiffusers) sao reaproveitadas aqui, com uma diferenca: os pesos podem estar em int8\n(torchao), entao a dobra faz dequant -> soma do delta -> re-quant, preservando o\n`nn.Parameter`. Em bf16 o comportamento e o do Space original (delta em float32).\n\nUso:\n    h3_lora.load(pipe.transformer, "larry", strength=1.0)   # carrega + dobra\n    h3_lora.set_active(pipe.transformer, "lightx")           # troca in place\n    h3_lora.set_active(pipe.transformer, "off")              # desdobra\n"""\n\nfrom __future__ import annotations\n\nimport os\n\nimport torch\n\nLARRY_REPO = "larryvrh/MiniMax-H3-Turbo-Lora"\nLARRY_FILE = "minimax_h3_turbo_4step_ema_ckpt850.safetensors"\nLIGHTX_REPO = "lightx2v/Minimax-h3-Turbo"\nLIGHTX_FILE = "minimax_h3_fl2v_turbo_4step_v0.1.safetensors"\nLIGHTX_ALPHA = 8  # alpha 8 / rank 128 = 0.0625\n\n# Modulos que ficam em bf16 (nunca em int8): embeddings, projecoes de patch/audio,\n# o norm_out final e o token_refiner. ~0.844 B de parametros.\nAO_SKIP = [\n    "proj_in",\n    "proj_out",\n    "norm_out.linear",\n    "context_embedder",\n    "time_embedder",\n    "token_refiner",\n    "audio_proj_in",\n    "audio_proj_out",\n]\n\n_STATE: dict = {"sets": None, "active": "off", "transformer": None}\n\n\ndef _repo_file(repo: str, preferred: str) -> str:\n    """Nome do .safetensors no repo, com fallback para o primeiro arquivo valido."""\n    from huggingface_hub import list_repo_files\n\n    try:\n        files = list_repo_files(repo)\n    except Exception:\n        return preferred\n    if preferred in files:\n        return preferred\n    candidates = [f for f in files if f.endswith(".safetensors")]\n    if not candidates:\n        raise FileNotFoundError(f"Nenhum .safetensors em {repo}: {files}")\n    for hint in ("ckpt850", "4step", "turbo"):\n        for f in candidates:\n            if hint in f:\n                return f\n    return candidates[0]\n\n\ndef _larry_targets(name: str, b: torch.Tensor, inner_dim: int):\n    """Mapeia a arvore de referencia para a arvore do diffusers, transformando as linhas."""\n    if name.startswith("token_refiner.blocks."):\n        target = name.replace("token_refiner.blocks.", "token_refiner.refiner_blocks.", 1)\n    elif name.startswith("blocks."):\n        target = name.replace("blocks.", "transformer_blocks.", 1)\n    else:\n        target = name\n    target = target.replace("final_layer.adaln_proj.linear", "norm_out.linear")\n\n    if target.endswith(".attn.qkv_proj"):\n        prefix = target.removesuffix("qkv_proj")\n        return [\n            (f"{prefix}to_{kind}.weight", part.contiguous())\n            for kind, part in zip(("q", "k", "v"), b.split(inner_dim, dim=0))\n        ]\n    if target.endswith(".mlp.fc1"):\n        gate, value = b.chunk(2, dim=0)\n        return [(target.replace(".mlp.fc1", ".ff.net.0.proj") + ".weight", torch.cat([value, gate]).contiguous())]\n    if target.endswith(".mlp.fc2"):\n        return [(target.replace(".mlp.fc2", ".ff.net.2") + ".weight", b)]\n    if target.endswith(".attn.out_proj"):\n        return [(target.replace(".attn.out_proj", ".attn.to_out.0") + ".weight", b)]\n    return [(target + ".weight", b)]\n\n\ndef _download(repo: str, preferred: str) -> str:\n    from huggingface_hub import hf_hub_download\n\n    filename = _repo_file(repo, preferred)\n    print(f"  baixando {repo}/{filename} ...", flush=True)\n    return hf_hub_download(repo, filename)\n\n\ndef _load_larry(inner_dim: int, scale: float) -> dict:\n    from safetensors.torch import load_file\n\n    lora = load_file(_download(LARRY_REPO, LARRY_FILE))\n    bases = sorted({key.rsplit(".lora_", 1)[0] for key in lora})\n    entries = []\n    for name in bases:\n        a = lora[f"{name}.lora_A.weight"]\n        b = lora[f"{name}.lora_B.weight"]\n        entries.extend((key, a, b_part) for key, b_part in _larry_targets(name, b, inner_dim))\n    return {"label": f"{LARRY_REPO}", "scale": scale, "entries": entries}\n\n\ndef _load_lightx() -> dict:\n    from safetensors.torch import load_file\n\n    lora = load_file(_download(LIGHTX_REPO, LIGHTX_FILE))\n    suffix_a, suffix_b = ".lora_A.default.weight", ".lora_B.default.weight"\n    bases = sorted({key[: -len(suffix_a)] for key in lora if key.endswith(suffix_a)})\n    ranks = {lora[f"{name}{suffix_a}"].shape[0] for name in bases}\n    if len(ranks) != 1:\n        raise ValueError(f"LoRAs de rank misto em {LIGHTX_FILE}: {sorted(ranks)}")\n    entries = [(f"{name}.weight", lora[f"{name}{suffix_a}"], lora[f"{name}{suffix_b}"]) for name in bases]\n    return {"label": f"{LIGHTX_REPO}", "scale": LIGHTX_ALPHA / ranks.pop(), "entries": entries}\n\n\ndef _is_int8(param: torch.Tensor) -> bool:\n    try:\n        from torchao.quantization import Int8Tensor\n\n        return isinstance(param.data, Int8Tensor)\n    except Exception:\n        return False\n\n\ndef _apply(entries, transformer, sign: float) -> None:\n    """Soma (ou subtrai, com sign=-1) o delta da LoRA dentro do peso."""\n    from torch.nn import Parameter\n    from torchao.quantization import Int8Tensor, PerRow\n\n    for key, a, b in entries:\n        parent = transformer\n        for part in key.split(".")[:-1]:\n            parent = getattr(parent, part, None)\n            if parent is None:\n                raise KeyError(f"alvo da LoRA `{key}` nao existe no transformer")\n        name = key.rsplit(".", 1)[-1]\n        param = getattr(parent, name, None)\n        if param is None or name not in parent._parameters:\n            raise KeyError(f"alvo da LoRA `{key}` nao existe no transformer")\n        delta = sign * (b.to(torch.float32) @ a.to(torch.float32))\n        data = param.data\n        if _is_int8(param):\n            # intermediario em bf16: corta o pico de memoria pela metade e a\n            # precisao ja e a do proprio int8 (~0.4%).\n            # o tensor re-quantizado tem que ser REINSERIDO em _parameters:\n            # `param.data = ...` e ignorado em silencio nos pesos combinados\n            # Int8Tensor+Parameter que o quantizador cria.\n            fp = data.dequantize(torch.bfloat16) + delta.to(data.device, torch.bfloat16)\n            novo = Int8Tensor.from_hp(fp, PerRow(dim=-1))\n            parent._parameters[name] = Parameter(novo, requires_grad=False)\n        else:\n            param.data = (data.to(torch.float32) + delta.to(data.device)).to(data.dtype)\n\n\ndef _fold_named(transformer, name: str) -> None:\n    state = _STATE\n    if state["active"] == name:\n        return\n    if state["active"] != "off":\n        old = state["sets"][state["active"]]\n        _apply(old["entries"], transformer, -old["scale"])\n    if name != "off":\n        new = state["sets"][name]\n        _apply(new["entries"], transformer, new["scale"])\n    state["active"] = name\n\n\ndef load(transformer, name: str = "larry", strength: float = 1.0) -> str:\n    """Carrega os dois LoRAs para a RAM e dobra o escolhido nos pesos."""\n    state = _STATE\n    if state["sets"] is None:\n        inner_dim = transformer.config.num_attention_heads * transformer.config.attention_head_dim\n        sets = {}\n        try:\n            sets["larry"] = _load_larry(inner_dim, float(strength))\n        except Exception as error:\n            print(f"  aviso: larry nao carregou ({error})")\n        try:\n            sets["lightx"] = _load_lightx()\n        except Exception as error:\n            print(f"  aviso: lightx nao carregou ({error})")\n        if not sets:\n            return "nenhum LoRA disponivel (rodando na base)"\n        state["sets"] = sets\n        state["active"] = "off"\n    if name not in state["sets"]:\n        name = sorted(state["sets"])[0] if name not in ("off", "") else "off"\n    state["transformer"] = transformer\n    _fold_named(transformer, name)\n    parts = ", ".join(f"{k} ({len(v[\'entries\'])} pesos, scale {v[\'scale\']:.4f})" for k, v in state["sets"].items())\n    return f"LoRAs carregados: {parts} · ativo: {state[\'active\']}"\n\n\ndef set_active(transformer, name: str) -> str:\n    """Troca a LoRA dobrada in place. \'off\' desdobra tudo."""\n    state = _STATE\n    if state["sets"] is None:\n        return "off"\n    if name not in state["sets"] and name not in ("off", ""):\n        print(f"LoRA \'{name}\' indisponivel; usando \'off\'")\n        name = "off"\n    if state["active"] != "off" and state["active"] not in state["sets"]:\n        state["active"] = "off"\n    _fold_named(transformer, name)\n    return state["active"]\n\n\ndef available() -> list:\n    return sorted(_STATE["sets"]) + ["off"] if _STATE["sets"] else ["off"]\n', encoding='utf-8')
(appdir / 'diario_app.py').write_text('"""MiniMax-H3 Diário: interface simples, modelos residentes na RAM e cache persistente."""\nimport gc\nimport hashlib\nimport json\nimport os\nimport secrets\nimport shutil\nimport threading\nimport time\nfrom fnmatch import fnmatch\nfrom pathlib import Path\n\nimport gradio as gr\nimport psutil\nfrom PIL import Image, ImageOps\n\nTEST = os.environ.get("H3_TEST_MODE") == "1"\nROOT = Path(os.environ.get("H3_WORK", "/content/h3_diario"))\nCACHE = Path(os.environ.get("H3_CACHE", str(ROOT / "pesos")))\nOUTPUT = Path(os.environ.get("H3_OUTPUT", str(ROOT / "videos")))\nOUTPUT.mkdir(parents=True, exist_ok=True)\nROOT.mkdir(parents=True, exist_ok=True)\nCANVASES = {"Vertical · 768×1344": (1344, 768), "Vertical rápido · 544×960": (960, 544),\n            "Horizontal · 1344×768": (768, 1344), "Horizontal rápido · 960×544": (544, 960),\n            "Quadrado · 768×768": (768, 768)}\nif TEST:\n    CANVASES = {"Teste · 128×128": (128, 128)}\n\n\ndef frames_for(seconds):\n    n = round(float(seconds) * 24)\n    while n % 17 != 5:\n        n += 1\n    return n\n\n\ndef fit(image, width, height):\n    if image is None:\n        return None\n    return ImageOps.fit(ImageOps.exif_transpose(image).convert("RGB"), (width, height),\n                        method=Image.Resampling.LANCZOS)\n\n\ndef image_key(image):\n    return hashlib.sha256(image.tobytes()).hexdigest() if image is not None else None\n\n\nclass Weights:\n    """Revisão fixa e verificação de TODOS os arquivos, incluindo shards, antes de usar."""\n    def __init__(self, root, local):\n        self.root, self.local = Path(root), Path(local)\n\n    @staticmethod\n    def complete(root, files):\n        return all((root / f["name"]).is_file() and\n                   (root / f["name"]).stat().st_size == f["size"] for f in files)\n\n    def ensure(self, repo, name, patterns, report):\n        from huggingface_hub import HfApi, snapshot_download\n        dest = self.root / name\n        dest.mkdir(parents=True, exist_ok=True)\n        manifest = dest / "diario_manifest.json"\n        if manifest.exists():\n            inventory = json.loads(manifest.read_text())\n        else:\n            pins_file = self.root / "revisions.json"\n            pins = json.loads(pins_file.read_text()) if pins_file.exists() else {}\n            info = HfApi().model_info(repo, revision=pins.get(repo), files_metadata=True)\n            files = [{"name": f.rfilename, "size": f.size} for f in info.siblings\n                     if any(fnmatch(f.rfilename, p) for p in patterns)]\n            if not files or any(f["size"] is None for f in files):\n                raise RuntimeError(f"Não consegui verificar os arquivos de {repo}.")\n            pins[repo] = info.sha\n            tmp = pins_file.with_suffix(".partial")\n            tmp.write_text(json.dumps(pins))\n            tmp.replace(pins_file)\n            inventory = {"revision": info.sha, "files": files}\n            tmp = manifest.with_suffix(".partial")\n            tmp.write_text(json.dumps(inventory))\n            tmp.replace(manifest)\n        files = inventory["files"]\n        if not self.complete(dest, files):\n            needed = sum(max(0, f["size"] - ((dest / f["name"]).stat().st_size\n                                            if (dest / f["name"]).is_file() else 0)) for f in files)\n            free = shutil.disk_usage(dest).free\n            if needed > free:\n                raise RuntimeError(f"Sem espaço para {name}: precisa aproximadamente {needed/2**30:.0f} GiB livres.")\n            report(f"Baixando {name} (primeira vez; interrupções podem ser retomadas)…")\n            snapshot_download(repo, revision=inventory["revision"], local_dir=dest,\n                              allow_patterns=[f["name"] for f in files], max_workers=4)\n            if not self.complete(dest, files):\n                raise RuntimeError(f"Download de {name} incompleto. Clique em Preparar novamente.")\n        else:\n            report(f"{name}: pesos salvos encontrados ✓")\n        staged = self.local / name\n        if dest.resolve() == staged.resolve():\n            return dest\n        staged.mkdir(parents=True, exist_ok=True)\n        missing = [f for f in files if not self.complete(staged, [f])]\n        if sum(f["size"] for f in missing) > shutil.disk_usage(staged).free:\n            raise RuntimeError("Disco local sem espaço para copiar os pesos do Drive. Use o local-scratch ou desative a cópia local.")\n        if missing:\n            report(f"Copiando {name} do Drive para o disco rápido da sessão…")\n        for f in missing:\n            target = staged / f["name"]\n            target.parent.mkdir(parents=True, exist_ok=True)\n            tmp = target.with_name(target.name + ".partial")\n            shutil.copyfile(dest / f["name"], tmp)\n            tmp.replace(target)\n        return staged\n\n\nclass Engine:\n    def __init__(self):\n        self.cond = self.pipe = self.manager = None\n        self.cancel = threading.Event()\n        self.log_lines = []\n        self.progress = None\n        self.step = 0\n        self.steps = 8\n        self.paths = None\n        self.pack = os.environ.get("H3_PACK", "1") == "1"\n        self.generator_bytes = 0\n        self.attention = None\n        self.weights = Weights(CACHE, ROOT / "pesos" if os.environ.get("H3_STAGE", "1") == "1" else CACHE)\n\n    def log(self, message):\n        self.log_lines.append(str(message))\n        self.log_lines = self.log_lines[-80:]\n        print(message, flush=True)\n        if self.progress:\n            self.progress(0, desc=str(message))\n\n    def checkpoint(self):\n        if self.cancel.is_set():\n            raise InterruptedError("Cancelado. Os pesos continuam salvos.")\n\n    def prepare(self):\n        import torch\n        if not TEST:\n            if not torch.cuda.is_available():\n                raise RuntimeError("Selecione GPU A100 no Colab antes de abrir o painel.")\n            vram = torch.cuda.get_device_properties(0).total_memory / 2**30\n            ram = psutil.virtual_memory().total / 2**30\n            if vram < 75 or ram < 140:\n                raise RuntimeError(f"Esta versão é para seu runtime de 80 GB GPU / 167 GB RAM. Encontrado: {vram:.0f}/{ram:.0f} GiB.")\n        if self.paths is None:\n            if TEST:\n                self.paths = (Path("/tmp/h3tiny1"), Path("/tmp/h3tiny"), {})\n            else:\n                patterns1 = ["modular_model_index.json", "model_index.json", "text_encoder/*", "tokenizer/*", "processor/*"]\n                patterns2 = ["modular_model_index.json", "model_index.json", "transformer/*", "vae/*", "audio_vae/*", "scheduler/*", "audio_scheduler/*"]\n                p1 = self.weights.ensure("MiniMaxAI/MiniMax-H3", "conditioner", patterns1, self.log)\n                self.checkpoint()\n                p2 = self.weights.ensure("MiniMaxAI/MiniMax-H3", "generator", patterns2, self.log)\n                from h3_lora import LARRY_REPO, LARRY_FILE, LIGHTX_REPO, LIGHTX_FILE\n                adapters = {}\n                for repo, fn, name in [(LARRY_REPO, LARRY_FILE, "larry"), (LIGHTX_REPO, LIGHTX_FILE, "lightx")]:\n                    self.checkpoint()\n                    folder = self.weights.ensure(repo, name, [fn], self.log)\n                    adapters[repo] = str(folder / fn)\n                self.paths = (p1, p2, adapters)\n        self.checkpoint()\n        if self.cond is None:\n            from h3_blocks import MiniMaxH3ConditionerBlocks\n            self.log("Carregando condicionador na RAM (uma vez por sessão)…")\n            cond = MiniMaxH3ConditionerBlocks().init_pipeline(str(self.paths[0]))\n            cond.load_components(names=["tokenizer", "processor", "text_encoder"],\n                                 pretrained_model_name_or_path=str(self.paths[0]), dtype=torch.bfloat16,\n                                 device_map={"text_encoder": {"": "cpu"}})\n            for name in ("text_encoder", "tokenizer", "processor"):\n                if getattr(cond, name) is None:\n                    raise RuntimeError(f"Não carregou {name}. Veja o log do Colab.")\n            self.cond = cond\n        self.checkpoint()\n        if self.pipe is None:\n            from diffusers import ComponentsManager, TorchAoConfig\n            from torchao.quantization import Int8WeightOnlyConfig\n            from h3_blocks import MiniMaxH3GeneratorBlocks\n            import h3_lora\n            self.log("Carregando gerador INT8 na RAM (uma vez por sessão)…")\n            manager = ComponentsManager()\n            pipe = MiniMaxH3GeneratorBlocks().init_pipeline(str(self.paths[1]), components_manager=manager, collection="diario")\n            pipe.load_components(names=["transformer", "scheduler", "audio_scheduler"],\n                                 pretrained_model_name_or_path=str(self.paths[1]),\n                                 dtype={"transformer": torch.bfloat16, "default": torch.float32},\n                                 quantization_config={"transformer": TorchAoConfig(Int8WeightOnlyConfig(), modules_to_not_convert=h3_lora.AO_SKIP)})\n            if pipe.transformer is None:\n                raise RuntimeError("Não carregou o transformer. Veja o log do Colab.")\n            if not any(h3_lora._is_int8(p) for p in pipe.transformer.parameters()):\n                raise RuntimeError("Quantização INT8 não aplicada; geração interrompida para evitar OOM.")\n            if not TEST:\n                h3_lora._download = lambda repo, filename: self.paths[2][repo]\n                self.log(h3_lora.load(pipe.transformer, "larry", strength=1.0))\n                if set(h3_lora.available()) != {"larry", "lightx", "off"}:\n                    raise RuntimeError("Uma das LoRAs falhou ao carregar. Veja o log.")\n            pipe.load_components(names=["vae", "audio_vae"], pretrained_model_name_or_path=str(self.paths[1]), dtype=torch.float32)\n            if pipe.vae is None or pipe.audio_vae is None:\n                raise RuntimeError("Não carregou as VAEs.")\n            pipe.transformer.register_forward_pre_hook(lambda m, args: self._step())\n            if not TEST:\n                self.generator_bytes = sum(\n                    p.numel() * (1 if h3_lora._is_int8(p) else p.element_size())\n                    for name in ("transformer", "vae", "audio_vae")\n                    for p in getattr(pipe, name).parameters())\n                try:\n                    pipe.transformer.set_attention_backend("_native_cudnn")\n                    self.attention = "_native_cudnn"\n                    self.log("atenção: cuDNN (mais rápida)")\n                except Exception as error:\n                    self.attention = None\n                    self.log(f"atenção: padrão ({type(error).__name__})")\n            self.pipe, self.manager = pipe, manager\n        self.log("Pronto. Modelos na RAM; pesos preservados para os próximos vídeos.")\n\n    def _step(self):\n        self.checkpoint()\n        self.step += 1\n        if self.progress:\n            self.progress(min(self.step / self.steps, 1), desc=f"Gerando · passo {self.step}/{self.steps}")\n\n    def park_generator(self):\n        import torch\n        if self.manager:\n            self.manager.disable_auto_cpu_offload()\n        if self.pipe:\n            for name in ("transformer", "vae", "audio_vae"):\n                getattr(self.pipe, name).to("cpu")\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n\n    def place_generator(self):\n        """Gerador inteiro na GPU (rápido); se não couber, usa offload com reserva."""\n        import torch\n        if TEST or self.pipe is None:\n            return\n        if self.pack:\n            if next(self.pipe.transformer.parameters()).device.type == "cuda":\n                return\n            free = torch.cuda.mem_get_info()[0]\n            if free < self.generator_bytes + 6 * 2**30:\n                self.log(f"pack: só {free/2**30:.0f} GiB livres; usando offload")\n                self.pack = False\n            else:\n                try:\n                    started = time.time()\n                    for name in ("transformer", "vae", "audio_vae"):\n                        getattr(self.pipe, name).to("cuda")\n                    torch.cuda.empty_cache()\n                    self.log(f"gerador na GPU em {time.time()-started:.0f}s · "\n                             f"livre {torch.cuda.mem_get_info()[0]/2**30:.0f} GiB")\n                    return\n                except Exception as error:\n                    self.log(f"pack falhou ({error}); usando offload")\n                    for name in ("transformer", "vae", "audio_vae"):\n                        try:\n                            getattr(self.pipe, name).to("cpu")\n                        except Exception:\n                            pass\n                    gc.collect()\n                    torch.cuda.empty_cache()\n                    self.pack = False\n        if self.manager is not None and not self.manager._auto_offload_enabled:\n            from h3_blocks import arm_vae_hooks\n            self.manager.enable_auto_cpu_offload(device="cuda", memory_reserve_margin="12GB")\n            arm_vae_hooks(self.pipe)\n\n    def condition(self, prompt, first, last, h, w, n, rewrite):\n        import torch\n        key = hashlib.sha256(json.dumps([prompt, image_key(first), image_key(last), h, w, n, rewrite]).encode()).hexdigest()\n        wires = CACHE / "prompts"\n        wires.mkdir(parents=True, exist_ok=True)\n        wire = wires / (key + ".pt")\n        if wire.exists():\n            self.log("Prompt e imagens já preparados: reutilizando condicionamento ✓")\n            return torch.load(wire, map_location="cpu", weights_only=True)\n        self.park_generator()\n        device = "cpu" if TEST else "cuda"\n        self.log("Preparando prompt e imagens…")\n        cond_started = time.time()\n        try:\n            if not TEST:\n                required = sum(p.numel() * p.element_size() for p in self.cond.text_encoder.parameters())\n                free = torch.cuda.mem_get_info()[0]\n                if free < required + 2 * 2**30:\n                    raise RuntimeError("GPU ocupada por outros modelos ou outra execução. Use uma sessão livre do Colab; os pesos e vídeos salvos serão preservados.")\n            self.cond.text_encoder.to(device)\n            if rewrite:\n                import prompt_rewrite\n                prompt = prompt_rewrite.refine_keyframe_prompt(self.cond, prompt, first, last, n,\n                                                               max_new_tokens=96 if TEST else 3072)\n            self.checkpoint()\n            state = self.cond(prompt=prompt, image=first, last_image=last, height=h, width=w)\n            payload = {"prompt_embeds": state.get("prompt_embeds").detach().cpu(),\n                       "text_token_tags": state.get("text_token_tags").detach().cpu(), "prompt": prompt}\n            del state\n            tmp = wire.with_suffix(".partial")\n            torch.save(payload, tmp)\n            tmp.replace(wire)\n            self.log(f"condicionador pronto em {time.time()-cond_started:.0f}s")\n            return payload\n        finally:\n            # Move o objeto vivo inteiro para CPU, mesmo se manager/callback retém referências.\n            self.cond.text_encoder.to("cpu")\n            gc.collect()\n            if torch.cuda.is_available():\n                torch.cuda.empty_cache()\n\n    def render(self, payload, first, last, h, w, n, steps, seed, lora):\n        import torch\n        import h3_lora\n        from diffusers.utils import encode_video\n        self.step, self.steps = 0, steps\n        self.checkpoint()\n        if lora != "off" and lora not in h3_lora.available():\n            raise RuntimeError(f"LoRA {lora} indisponível.")\n        if h3_lora._STATE["active"] != lora:\n            self.park_generator()\n        h3_lora.set_active(self.pipe.transformer, lora)\n        self.place_generator()\n        device = "cpu" if TEST else "cuda"\n        if not TEST:\n            torch.cuda.reset_peak_memory_stats()\n        started = time.time()\n        attempts = 3\n        while True:\n            try:\n                state = self.pipe(prompt_embeds=payload["prompt_embeds"].to(device), text_token_tags=payload["text_token_tags"],\n                                  image=first, last_image=last, height=h, width=w, num_frames=n,\n                                  num_inference_steps=steps, generator=torch.Generator("cpu").manual_seed(seed))\n                break\n            except Exception as error:\n                message = str(error).lower()\n                if attempts > 1 and self.attention == "_native_cudnn" and ("scaled_dot_product" in message or "backend" in message):\n                    self.log("cuDNN falhou; repetindo com a atenção padrão")\n                    self.attention = None\n                    self.pipe.transformer.reset_attention_backend()\n                    attempts -= 1\n                    continue\n                if attempts > 1 and self.pack and "out of memory" in message:\n                    self.log("VRAM estourou com o pack; voltando para offload e repetindo")\n                    self.pack = False\n                    self.park_generator()\n                    self.place_generator()\n                    attempts -= 1\n                    continue\n                raise\n        done = time.time()\n        path = OUTPUT / f"h3_{time.strftime(\'%Y%m%d_%H%M%S\')}_{seed}_{secrets.token_hex(3)}.mp4"\n        encode_video(state.get("videos")[0], fps=24, output_path=str(path),\n                     audio=state.get("audio")[0].cpu(), audio_sample_rate=state.get("sampling_rate", 32000))\n        encoded = time.time()\n        peak = torch.cuda.max_memory_allocated() / 2**30 if torch.cuda.is_available() else 0\n        self.log(f"tempo: denoise+decode {done-started:.0f}s · encode {encoded-done:.0f}s · pico VRAM {peak:.1f} GiB")\n        del state\n        gc.collect()\n        if torch.cuda.is_available():\n            torch.cuda.empty_cache()\n        return str(path)\n\n\nengine = Engine()\n\n\ndef prepare_ui(progress=gr.Progress()):\n    engine.cancel.clear()\n    engine.progress = progress\n    try:\n        engine.prepare()\n        return "✅ Pronto para gerar.", "\\n".join(engine.log_lines)\n    except Exception as error:\n        engine.log(str(error))\n        raise gr.Error(str(error)) from error\n    finally:\n        engine.progress = None\n\n\ndef generate(prompt, first, last, canvas, duration, steps, seed, rewrite, turbo, count, progress=gr.Progress()):\n    if not prompt or not prompt.strip():\n        raise gr.Error("Escreva o prompt do vídeo.")\n    if canvas not in CANVASES or not 2 <= float(duration) <= 14:\n        raise gr.Error("Canvas ou duração inválida.")\n    if not 1 <= int(count) <= 4 or not 2 <= int(steps) <= 40:\n        raise gr.Error("Quantidade ou passos inválidos.")\n    seed = int(seed)\n    if seed < -1 or seed >= 2**63 - 4:\n        raise gr.Error("Seed deve ser -1 (aleatória) ou um inteiro positivo menor que 2⁶³−4.")\n    lora = {"Turbo larry (padrão)": "larry", "Turbo lightx": "lightx", "Desligado (base)": "off"}[turbo]\n    if lora == "off" and int(steps) < 28 and not TEST:\n        raise gr.Error("Sem Turbo, use ao menos 28 passos nas opções avançadas.")\n    engine.cancel.clear()\n    engine.progress = progress\n    created = []\n    refined = ""\n    try:\n        yield None, [], "Preparando…", "", "\\n".join(engine.log_lines)\n        engine.prepare()\n        h, w = CANVASES[canvas]\n        first, last = fit(first, w, h), fit(last, w, h)\n        n = frames_for(duration)\n        payload = engine.condition(prompt.strip(), first, last, h, w, n, rewrite)\n        refined = payload["prompt"]\n        base_seed = secrets.randbelow(2**31) if seed == -1 else seed\n        for i in range(int(count)):\n            engine.checkpoint()\n            engine.log(f"Vídeo {i+1}/{int(count)} · {w}×{h} · {n/24:.2f}s · seed {base_seed+i}")\n            path = engine.render(payload, first, last, h, w, n, int(steps), base_seed+i, lora)\n            created.append(path)\n            yield path, created[:], f"✅ {i+1}/{int(count)} concluído · seed {base_seed+i} · {n/24:.2f}s", refined, "\\n".join(engine.log_lines)\n    except InterruptedError as error:\n        yield created[-1] if created else None, created, str(error), refined, "\\n".join(engine.log_lines)\n    except Exception as error:\n        engine.log(f"Erro: {error}")\n        engine.park_generator()\n        raise gr.Error(f"{error}\\nPesos preservados. Consulte o log e tente novamente.") from error\n    finally:\n        engine.progress = None\n\n\ndef cancel_ui():\n    engine.cancel.set()\n    return "Cancelamento solicitado; termina no próximo ponto seguro."\n\n\ndef release_ui():\n    engine.park_generator()\n    return "GPU liberada. Modelos continuam na RAM e pesos no disco."\n\n\ndef history_ui():\n    return [str(p) for p in sorted(OUTPUT.glob("*.mp4"), reverse=True)]\n\n\ndef build_ui():\n    css = "#generate {min-height:52px;font-size:18px} .gradio-container {max-width:1200px!important}"\n    with gr.Blocks(title="MiniMax-H3 Diário") as demo:\n        gr.Markdown("# 🎬 MiniMax-H3\\nVídeo com áudio · texto, imagem inicial ou inicial + final")\n        with gr.Row():\n            with gr.Column():\n                with gr.Row():\n                    first = gr.Image(label="Imagem inicial (opcional)", type="pil", height=230)\n                    last = gr.Image(label="Imagem final (opcional)", type="pil", height=230)\n                prompt = gr.Textbox(label="Descreva o vídeo", lines=5,\n                                    placeholder=\'Cena, movimento, câmera e fala. Ex.: ela olha para a câmera e diz: <d>[Portuguese] Olá!</d>\')\n                with gr.Row():\n                    canvas = gr.Dropdown(list(CANVASES), value=next(iter(CANVASES)), label="Formato")\n                    duration = gr.Slider(2, 14, value=5, step=.5, label="Duração (s)")\n                rewrite = gr.Checkbox(value=True, label="Melhorar prompt automaticamente (Context-IR)")\n                with gr.Accordion("Opções avançadas", open=False):\n                    turbo = gr.Dropdown(["Turbo larry (padrão)", "Turbo lightx", "Desligado (base)"],\n                                        value="Desligado (base)" if TEST else "Turbo larry (padrão)", label="Turbo")\n                    steps = gr.Slider(2, 40, value=8, step=1, label="Passos (8 recomendado com Turbo)")\n                    seed = gr.Number(value=-1, precision=0, label="Seed (-1 = aleatória)")\n                    count = gr.Slider(1, 4, value=1, step=1, label="Variações do mesmo vídeo")\n                run = gr.Button("🎬 Gerar vídeo", variant="primary", elem_id="generate")\n                stop = gr.Button("Cancelar")\n            with gr.Column():\n                video = gr.Video(label="Seu vídeo", height=400)\n                files = gr.File(label="Baixar vídeos", file_count="multiple")\n                status = gr.Textbox(label="Status", value="Clique em Preparar ou Gerar para iniciar.", interactive=False)\n                with gr.Accordion("Prompt final e detalhes", open=False):\n                    refined = gr.Textbox(label="Prompt utilizado", lines=5, interactive=False)\n                    logs = gr.Textbox(label="Log", lines=8, interactive=False)\n        with gr.Accordion("Sessão e vídeos salvos", open=False):\n            prepare = gr.Button("Preparar modelos (uma vez por sessão)")\n            release = gr.Button("Liberar GPU")\n            refresh = gr.Button("Atualizar vídeos salvos")\n            history = gr.File(label="Histórico para download", file_count="multiple")\n            gr.Markdown(f"**Vídeos:** `{OUTPUT}`\\n\\n**Cache:** `{CACHE}`\\n\\n"\n                        "Drive conserva os pesos e vídeos entre sessões. Cache local termina com o runtime. "\n                        "Os vídeos continuam sendo salvos mesmo se você fechar o link. "\n                        "O Colab precisa permanecer conectado; desligue o runtime quando encerrar o trabalho.")\n        run.click(generate, [prompt, first, last, canvas, duration, steps, seed, rewrite, turbo, count],\n                  [video, files, status, refined, logs], concurrency_id="gpu", concurrency_limit=1)\n        stop.click(cancel_ui, outputs=status, queue=False)\n        prepare.click(prepare_ui, outputs=[status, logs], concurrency_id="gpu", concurrency_limit=1)\n        release.click(release_ui, outputs=status, concurrency_id="gpu", concurrency_limit=1)\n        refresh.click(history_ui, outputs=history, queue=False)\n    return demo, css\n\n\nif __name__ == "__main__":\n    import torch\n    from diffusers.modular_pipelines.minimax_h3.modular_pipeline import MiniMaxH3ModularPipeline\n    MiniMaxH3ModularPipeline.min_duration = property(lambda self: 2.0)\n    demo, css = build_ui()\n    demo.queue(max_size=8, default_concurrency_limit=1)\n    _, local, public = demo.launch(share=not TEST, server_name="127.0.0.1", server_port=int(os.environ.get("H3_PORT", "7860")),\n                                  prevent_thread_lock=True, allowed_paths=[str(OUTPUT)], css=css, show_error=True)\n    (ROOT / "link.json").write_text(json.dumps({"local": local, "public": public}))\n    while True:\n        time.sleep(60)\n', encoding='utf-8')

#@title 2 · Abrir painel
USE_DRIVE = True #@param {type:"boolean"}
COPY_TO_FAST_DISK = True #@param {type:"boolean"}

import json, os, shutil, subprocess, sys, time
from pathlib import Path
from IPython.display import display, HTML
import psutil

# Seleciona o disco com mais espaço; aproveita local-scratch quando disponível.
candidates = [Path('/content')]
for p in psutil.disk_partitions(all=True):
    mount = Path(p.mountpoint)
    if p.fstype in ('tmpfs', 'devtmpfs', 'proc', 'sysfs', 'squashfs', 'cgroup', 'cgroup2'):
        continue
    if str(mount).startswith(('/proc', '/sys', '/dev', '/etc', '/content/drive')):
        continue
    if mount.is_dir() and os.access(mount, os.W_OK):
        candidates.append(mount)
for p in ('/mnt/local-scratch', '/content/local-scratch', '/local-scratch', '/mnt/disks/local-scratch'):
    mount = Path(p)
    if mount.is_mount() and os.access(mount, os.W_OK):
        candidates.append(mount)
work = max(candidates, key=lambda p: shutil.disk_usage(p).free) / 'h3_diario'
work.mkdir(parents=True, exist_ok=True)

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    base = Path('/content/drive/MyDrive/MiniMax_H3_Diario')
    cache, output = base / 'pesos', base / 'videos'
    print('Cache persistente no Drive: reserve aproximadamente 150 GB livres.')
else:
    cache, output = work / 'pesos', work / 'videos'
    print('Cache local: reusa os pesos nesta sessão; eles serão perdidos ao encerrar o runtime.')
cache.mkdir(parents=True, exist_ok=True)
output.mkdir(parents=True, exist_ok=True)
print(f'Disco rápido: {work} · {shutil.disk_usage(work).free/2**30:.0f} GiB livres')

env = os.environ.copy()
env.update(H3_WORK=str(work), H3_CACHE=str(cache), H3_OUTPUT=str(output),
           H3_STAGE='1' if COPY_TO_FAST_DISK else '0',
           PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True', HF_HOME=str(work / 'hf_cache'),
           GRADIO_ANALYTICS_ENABLED='False', PYTHONUNBUFFERED='1')
log_path = work / 'painel.log'
link_file = work / 'link.json'
if 'h3_process' not in globals() or h3_process.poll() is not None:
    link_file.unlink(missing_ok=True)
    h3_log = open(log_path, 'w')
    h3_process = subprocess.Popen([sys.executable, '/content/h3_diario_app/diario_app.py'],
                                  env=env, stdout=h3_log, stderr=subprocess.STDOUT)
    for _ in range(180):
        if h3_process.poll() is not None:
            raise RuntimeError('O painel não abriu. Log:\n' + log_path.read_text()[-12000:])
        if link_file.exists():
            break
        time.sleep(1)
    else:
        print(log_path.read_text()[-12000:])
        raise RuntimeError('O link ainda não abriu. Rode esta célula novamente; o processo continua vivo.')
links = json.loads(link_file.read_text())
if not links.get('public'):
    raise RuntimeError('O túnel Gradio não abriu. Veja o log: ' + str(log_path))
display(HTML('<h2>✅ Painel aberto</h2><a href="' + links['public'] + '" target="_blank">'
             '<b>CLIQUE AQUI PARA GERAR SEUS VÍDEOS</b></a>'))
print('No painel, clique em Preparar uma vez. Depois use apenas a interface.')
print('O link funciona enquanto este runtime estiver conectado. Log:', log_path)


Mounted at /content/drive
Cache persistente no Drive: reserve aproximadamente 150 GB livres.
Disco rápido: /mnt/local-scratch/h3_diario · 349 GiB livres


No painel, clique em Preparar uma vez. Depois use apenas a interface.
O link funciona enquanto este runtime estiver conectado. Log: /mnt/local-scratch/h3_diario/painel.log
